# 01 - Ekstraksi fitur (CelebA-Spoof subset)
Face anti-spoofing, UTS Foundation of ML & Optimization (UDINUS). Dijalankan di Google Colab (GPU).

Alur: ekstrak arsip dataset dari Google Drive, bangun indeks berkas, audit berkas, pembagian per subjek 70/15/15 (seed 42),
lalu ekstraksi fitur MobileNetV3-Small ImageNet (576-d) dan fitur FFT radial (56) + LBP (10). Keluaran: `features.npz`, `file_index_clean.csv`.

Jalur data (`/content/drive/MyDrive/Dataset_CelebA-Spoof/...`) khusus Drive penulis; ubah `WORK` dan jalur arsip bila dijalankan ulang.
Sel pertama dan kedua diedit setelah dijalankan (rapikan perintah apt ganda, `find` untuk `.png`, tambah catatan); keluaran sel pertama lama dihapus. Logika tidak diubah.

In [ ]:
!apt-get -qq update && apt-get -qq install -y libarchive-tools
!which bsdtar
from google.colab import drive; drive.mount('/content/drive')
import subprocess, glob, os

parts = sorted(glob.glob('/content/drive/MyDrive/Dataset_CelebA-Spoof/*zip.00[1-4]'))
print(parts)

os.makedirs('/content/celeba', exist_ok=True)
p = subprocess.Popen(['bsdtar', '-xf', '-'], stdin=subprocess.PIPE,
                     stderr=subprocess.PIPE, cwd='/content/celeba')
try:
    for part in parts:
        with open(part, 'rb') as fh:
            while chunk := fh.read(1 << 24):
                p.stdin.write(chunk)
    p.stdin.close()
except BrokenPipeError:
    pass
print(p.stderr.read().decode()[-600:])
p.wait()

!find /content/celeba -name "*.png" | wc -l
!ls /content/celeba | head

In [ ]:
import os, glob, pandas as pd

root = '/content/celeba/CelebA_Spoof/Data'
# CATATAN: pola '*.*' ikut menangkap berkas pendamping *_BB.txt (kotak wajah).
# Berkas itu terbuang oleh filter ukuran di sel berikutnya (8.526 dari 8.541 berkas 'dibuang'),
# sehingga 8.512 citra akhir tidak terpengaruh. Untuk run ulang, pola '*.png' memberi hitungan awal yang bersih.
rows = []
for p in glob.glob(f'{root}/*/*/*/*.*'):
    sp, subj, label, fn = p.split('/')[-4:]
    rows.append((p, sp, int(subj), label, os.path.getsize(p)))
df = pd.DataFrame(rows, columns=['path','split','subject','label','bytes'])

print(df.shape)
print(df.groupby('label').size())
print('subjek:', df.subject.nunique(), '| rentang:', df.subject.min(), '-', df.subject.max())
print(df.groupby('label').bytes.describe()[['mean','min','max']])
print(df.groupby('subject').label.value_counts().unstack(fill_value=0).describe().loc[['mean','min','max']])

os.makedirs('/content/drive/MyDrive/Dataset_CelebA-Spoof/work', exist_ok=True)
df.drop(columns='bytes').to_csv('/content/drive/MyDrive/Dataset_CelebA-Spoof/work/file_index.csv', index=False)

(17053, 5)
label
live      3402
spoof    13651
dtype: int64
subjek: 79 | rentang: 3613 - 10169
                mean   min         max
label                                 
live   343196.071429  19.0  29298391.0
spoof  230525.827046  19.0    759388.0
label       live       spoof
mean   43.063291  172.797468
min     0.000000   18.000000
max    62.000000  594.000000


In [ ]:
!pip -q install scikit-image
import torch, torchvision, numpy as np, pandas as pd, os
from PIL import Image
from skimage.feature import local_binary_pattern
from torch.utils.data import Dataset, DataLoader

WORK = '/content/drive/MyDrive/Dataset_CelebA-Spoof/work'
df = pd.read_csv(f'{WORK}/file_index.csv')
df['y'] = (df.label == 'spoof').astype(int)          # 1 = serangan (spoof)

# 1) Audit: buang file rusak / outlier ukuran
df['bytes'] = df.path.map(os.path.getsize)
bad = df[(df.bytes < 2000) | (df.bytes > 8_000_000)]
print('dibuang:', len(bad), bad.label.value_counts().to_dict())
df = df.drop(bad.index).reset_index(drop=True)

# 2) Split PER SUBJEK (70/15/15), seed tetap agar bisa direplikasi
rng = np.random.RandomState(42)
ids = rng.permutation(df.subject.unique()); n = len(ids)
tr, va = set(ids[:int(.70*n)]), set(ids[int(.70*n):int(.85*n)])
df['split2'] = df.subject.map(lambda s: 'train' if s in tr else 'val' if s in va else 'test')
print(df.groupby(['split2', 'label']).size().unstack())

# 3) Fitur frekuensi: profil radial FFT (56) + histogram LBP (10)
def freq_feat(g):
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))
    yy, xx = np.indices(mag.shape); r = np.hypot(yy - 112, xx - 112).astype(int).ravel()
    prof = np.bincount(r, mag.ravel()) / np.maximum(np.bincount(r), 1)
    return prof[:112].reshape(56, 2).mean(1)
def lbp_feat(g):
    l = local_binary_pattern(g, 8, 1, 'uniform')
    return np.bincount(l.astype(int).ravel(), minlength=10) / l.size

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32); std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
class DS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        try: im = Image.open(s.p[i]).convert('RGB').resize((224, 224)); ok = 1
        except Exception: im = Image.new('RGB', (224, 224)); ok = 0
        a = np.asarray(im, dtype=np.float32) / 255.
        g = np.asarray(im.convert('L'))
        ff = np.concatenate([freq_feat(g.astype(np.float32)), lbp_feat(g)]).astype(np.float32)
        t = torch.from_numpy(((a - mean) / std).transpose(2, 0, 1).astype(np.float32))
        return t, torch.from_numpy(ff), ok

# 4) Fitur CNN: MobileNetV3-small ImageNet (tanpa training ulang), 576 dimensi
dev = 'cuda'
m = torchvision.models.mobilenet_v3_small(weights='DEFAULT'); m.classifier = torch.nn.Identity(); m = m.to(dev).eval()
dl = DataLoader(DS(df.path), batch_size=128, num_workers=2, shuffle=False)
Xc, Xf, OK = [], [], []
with torch.no_grad():
    for i, (x, f, ok) in enumerate(dl):
        Xc.append(m(x.to(dev)).cpu().numpy()); Xf.append(f.numpy()); OK.append(ok.numpy())
        if i % 20 == 0: print(i, '/', len(dl))

np.savez_compressed(f'{WORK}/features.npz', Xc=np.vstack(Xc), Xf=np.vstack(Xf), ok=np.concatenate(OK),
                    y=df.y.values, subject=df.subject.values, split=df.split2.values)
df.drop(columns='bytes').to_csv(f'{WORK}/file_index_clean.csv', index=False)
print('selesai', np.vstack(Xc).shape, np.vstack(Xf).shape)

dibuang: 8541 {'spoof': 6825, 'live': 1716}
label   live  spoof
split2             
test     274   1168
train   1171   5023
val      241    635
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 120MB/s]


0 / 67
20 / 67
40 / 67
60 / 67
selesai (8512, 576) (8512, 66)


In [ ]:
print(bad.path.str.extract(r'\.(\w+)$')[0].value_counts())

0
txt    8526
png      15
Name: count, dtype: int64


In [ ]:
   import sys, torch, torchvision, sklearn, skimage, numpy, pandas, scipy
   print(sys.version.split()[0], torch.__version__, torchvision.__version__, sklearn.__version__, skimage.__version__, numpy.__version__, pandas.__version__, scipy.__version__)

3.13.15 2.11.0+cu130 0.26.0+cu130 1.6.1 0.25.2 2.1.3 2.2.3 1.16.3
